In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

import matplotlib.pyplot as plt

📘 Item–Item Collaborative Filtering Algorithm

Input:

Ratings dataset (User-ID, ISBN, Book-Rating).

Steps:

Build User–Item Matrix

Rows = Users, Columns = Books, Values = Ratings.

Compute Item–Item Similarity

Each column = a “vector” of ratings for that book.

Compare book vectors (using cosine similarity or correlation) → gives us an Item–Item Similarity Matrix.

Generate Recommendations for a User

Take books the user has already rated.

For each such book, look up its most similar books.

Weight similarity scores by how strongly the user rated the original book.

Aggregate scores across all rated books.

Filter out books the user already rated.

Return Top-N items with the highest score.

In [28]:
books = pd.read_csv("processed datasets/books.csv")

ratings = pd.read_csv("processed datasets/ratings.csv")

users = pd.read_csv("processed datasets/users.csv")


In [29]:
books = books.iloc[: ,:5]
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002-01-01,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001-01-01,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991-01-01,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999-01-01,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999-01-01,W. W. Norton &amp; Company


In [30]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [31]:
users.head()

,User-ID,Location,Age
0,2,"stockton, california, usa",18.0
1,4,"porto, v.n.gaia, portugal",17.0
2,6,"santa monica, california, usa",61.0
3,10,"albacete, wisconsin, spain",26.0
4,11,"melbourne, victoria, australia",14.0


In [32]:
books.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271351 entries, 0 to 271350
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271351 non-null  object
 1   Book-Title           271351 non-null  object
 2   Book-Author          271351 non-null  object
 3   Year-Of-Publication  271351 non-null  object
 4   Publisher            271351 non-null  object
dtypes: object(5)
memory usage: 10.4+ MB


In [33]:
users.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167730 entries, 0 to 167729
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   167730 non-null  int64  
 1   Location  167730 non-null  object 
 2   Age       167730 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 3.8+ MB


In [34]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1149780 non-null  int64 
 1   ISBN         1149780 non-null  object
 2   Book-Rating  1149780 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 26.3+ MB


In [35]:
##Due to high coputational costr we will take only dataset with more than 5star ratings

ratings = ratings [ratings['Book-Rating']  > 5]


In [36]:
print(ratings.shape , books.shape)

(363268, 3) (271351, 5)


#### choose only first 1000 users

In [37]:
ratings = ratings[ratings['User-ID'].isin(ratings['User-ID'].unique()[:1000])]

In [38]:
# Remove ratings for ISBNs not in books but in ratings
valid_isbns = set(books['ISBN'].unique())
invalid_rows = ~ratings['ISBN'].isin(valid_isbns)
ratings = ratings[~invalid_rows].reset_index(drop=True)

deleted = invalid_rows.sum()
print(f'Total deleted  : {deleted}')

Total deleted  : 518


In [39]:
books_isbn = set(books['ISBN'].unique())

ratings_isbn = set(ratings['ISBN'].unique())

extra_books = books_isbn - ratings_isbn
# print(extra_books.len())

extra_books_index = books[books['ISBN'].isin(extra_books)].index.tolist()

books.drop(extra_books_index , inplace=True)

In [40]:
print(ratings.shape , books.shape)

(3185, 3) (2917, 5)


In [41]:
#lets make the 

##### we have large dataset for users and books ,this will create large matrix so we will use item-item instead od user-user


## customer who like this also liked this

In [42]:
ratings.isna().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [43]:
rating_subset = ratings[ratings['User-ID'].isin(ratings['User-ID'].unique()[:10000])]

In [44]:
# Create user-item matrix (pivot table)
pt = rating_subset.pivot_table(index='User-ID', columns='ISBN', values='Book-Rating')


In [45]:
pt

ISBN,0002240114,000225669X,0002740230,000636988X,000649840X,0006543545,0007100221,0020259700,0020418809,0020427859,...,8817106100,881715010X,8817853712,8838910987,887641486X,9068340344,9500703203,950491036X,9681500555,9724115380
User-ID,,,,,,,,,,,,,,,,,,,,,
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278846,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
pt = pt.fillna(0)

item_similarities = cosine_similarity(pt.T)

item_similarities_df = pd.DataFrame(data = item_similarities , columns = pt.columns , index=pt.columns)
item_similarities_df

ISBN,0002240114,000225669X,0002740230,000636988X,000649840X,0006543545,0007100221,0020259700,0020418809,0020427859,...,8817106100,881715010X,8817853712,8838910987,887641486X,9068340344,9500703203,950491036X,9681500555,9724115380
ISBN,,,,,,,,,,,,,,,,,,,,,
0002240114,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000225669X,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0002740230,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000636988X,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000649840X,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9068340344,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9500703203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
950491036X,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


steps :
1) find users rated books
2) find similar books to users each  rated books
3) multiply with book rating to show imp of book
4) sort and show top 

In [91]:
def recommend_books_for_user(user_id, top_n=5):
    
    if user_id not in pt.index:
        print('USer not found')
    
    users_books = pt.loc[user_id]
    rated_books = users_books[users_books > 0]

    print(f'users Rated books : \n {rated_books}')

    similarity_score = {}

    for book , rating in rated_books.items():
        if book:
            similar_books = item_similarities_df.loc[book]
            similar_books = similar_books[(similar_books > 0) ]


            for isbn , similarity in similar_books.items():
                if isbn not in rated_books.keys().tolist():
                   # print(isbn , similarity)
                    similarity_score[isbn]  = similarity * rating

            

            recommendation = [(i , isbn) for i , isbn in similarity_score.items()]            

            recommendation.sort(reverse=True , key=lambda  x : x[1])
            

            top_recommendations = [isbn for isbn, score in recommendation[:top_n]]
            
            return books[books['ISBN'].isin(top_recommendations)]


In [92]:
recommend_books_for_user(276747)

users Rated books : 
 ISBN
0060517794    9.0
0671537458    9.0
0679776818    8.0
0943066433    7.0
1885408226    7.0
Name: 276747, dtype: float64


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
2342,0553274503,Flowers for Algernon (Bantam Classic),DANIEL KEYES,1984-01-01,Bantam
4020,0440211263,Circle of Friends,Maeve Binchy,1991-01-01,Dell
5070,014028009X,Bridget Jones's Diary,Helen Fielding,1999-01-01,Penguin Books
22430,0385260075,Cat's Eye,Margaret Atwood,1989-01-01,Doubleday Books
139023,0435120425,New Windmills: A Town Like Alice (New Windmills),Nevil Shute,1968-01-01,Heinemann Educational Books - Secondary Division


In [50]:
users = rating_subset['User-ID'].unique()[:10]
users

array([276729, 276744, 276747, 276748, 276751, 276754, 276762, 276772,
       276774, 276786], dtype=int64)

#### user-user based collabrative

In [52]:
user_item_matrix = pt.fillna(0)

# Compute cosine similarity between users (rows)
user_similarities = cosine_similarity(user_item_matrix)

user_similarities_df = pd.DataFrame(user_similarities, index=user_item_matrix.index, columns=user_item_matrix.index)
user_similarities_df

User-ID,8,9,12,14,16,17,19,22,26,32,...,278831,278832,278836,278843,278844,278846,278849,278851,278852,278854
User-ID,,,,,,,,,,,,,,,,,,,,,
8,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278846,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
278849,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
278851,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


steps
1) find similar user
2) llok at ratings from similar users
3) multiply ratings by each users similaritties
4) sort them

In [124]:
def rec_book_for_user(user_id , top_n = 5):

    if user_id not in user_similarities_df.index:
        print('User not found ')

    user_books = pt.loc[user_id]
    rated_books = user_books[user_books > 0]

    similar_users = user_similarities_df.loc[user_id]
    similar_users = similar_users[similar_users > 0]
    print(f"similar users : \n {similar_users}")

    similar_books = {}

    for user , similarity in similar_users.items():
        #exclude the original user(self)
        if user != user_id:        
            books_read = pt.loc[user]
            book_read = books_read[books_read > 0]

            # print(user , book_read)

            for isbn , rating in book_read.items():
                #make sure the books read by users are excluded
                if isbn not in rated_books.keys().tolist():
                    similar_books[isbn] = rating * similarity
            

            recommend_list = [(isbn , simi) for isbn , simi in similar_books.items() ]
            
            recommend_list.sort(key=lambda x : x[1] , reverse= True)

            top_recommends = [isbn for isbn , i in recommend_list[:top_n]]

            return books[books['ISBN'].isin(top_recommends)]

            
        


       


In [125]:
rec_book_for_user(276747)

similar users : 
 User-ID
276747    1.000000
278843    0.103752
Name: 276747, dtype: float64


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
403,0060173289,Divine Secrets of the Ya-Ya Sisterhood : A Novel,Rebecca Wells,1996-01-01,HarperCollins
712,0142000205,Icy Sparks,Gwyn Hyman Rubio,2001-01-01,Penguin Books
2969,0399146431,The Bonesetter's Daughter,Amy Tan,2001-01-01,Putnam Publishing Group
4666,0767902890,The Things They Carried,TIM O'BRIEN,1998-01-01,Broadway
137638,0590045296,Frog and Toad Are Friends,Arnold Lobel,1979-01-01,Scholastic Paperbacks (Mm)


## NOTES

#### 📘 Item–Item Collaborative Filtering (CF)

Similarity source: Ratings/behavior data.

Idea Two books are similar if the same users tend to rate them similarly.

**Example**:
Many users who rated Harry Potter also rated Percy Jackson.

Even if the metadata (author, genre, title) is different, system says: *"These books go together."*

**Strengths**:
- Learns from collective wisdom of users

**Weaknesses**:
- Cold start problem (new books with no ratings cannot be recommended)

📘 User–User Collaborative Filtering (CF)
1. What it is

We compare users instead of items.

Two users are considered similar if they rate books in a similar way.

Example:

User A and User B both gave 5★ to Harry Potter and 4★ to Percy Jackson.

System: “These users are alike → recommend books one liked but the other hasn’t read yet.

Great — this is the heart of understanding **Item–Item Collaborative Filtering**.
Let’s break it step by step, with an example.

---

## 🔑 Why do we create a **User–Item Matrix**?

### 1. What it is:

* A table where **rows = users**, **columns = items (books)**, and **values = ratings**.
* Example:

| User \ Book | Book A | Book B | Book C | Book D |
| ----------- | ------ | ------ | ------ | ------ |
| User 1      | 5      | 4      | NaN    | NaN    |
| User 2      | 4      | NaN    | 3      | 2      |
| User 3      | NaN    | 2      | 5      | NaN    |

(`NaN` means the user has not rated that book yet.)

---

### 2. Purpose:

* This matrix **connects users and items**.
* Even though we are doing **Item–Item CF**, we need to know:

  * Which items were rated by the same users.
  * How strongly ratings of two books correlate across all users.

---

### 3. How similarity comes from it:

* To know if **Book A** and **Book B** are similar, we check:

  * Did many users rate both?
  * Did they give similar ratings?

From the example:

* User 1 rated Book A = 5, Book B = 4
* User 2 rated Book A = 4, Book B = NaN
* User 3 rated Book A = NaN, Book B = 2

So Book A and Book B have overlapping ratings from User 1 (5 vs 4).
That overlap → contributes to similarity score.

---

### 4. Conceptual Meaning

* **Rows (users)**: tell us how people behave.
* **Columns (books)**: tell us what items were evaluated.
* By comparing **columns**, we discover **items that “behave alike” across the crowd**.

---

### 5. Why it’s necessary

Without the matrix:

* We can’t compute co-rating patterns (the core of collaborative filtering).
* Content-based doesn’t need it, because it uses metadata.
* But Item–Item CF depends **only on ratings** → the User–Item matrix is the foundation.

---

👉 In short:
The **User–Item Matrix** is the bridge that lets us compare items indirectly, through the users who rated them.

---

Do you want me to also **show the math of how similarity is computed from this matrix** (like cosine similarity between two book rating vectors)?
